In [ ]:
%pip install rdflib

  Using cached rdflib-7.6.0-py3-none-any.whl.metadata (12 kB)
Using cached rdflib-7.6.0-py3-none-any.whl (615 kB)



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\lenovo\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Open this file and check how many triples are there to see if it can run properly.

In [1]:
from rdflib import Graph

url= ("https://raw.githubusercontent.com/Timoseee/windows-cyber-threat-kg/main/windows-cyber-threat-kg.ttl")
g = Graph()
g.parse(url,format="turtle")

print(f"Loaded {len(g)} RDF triples.")

Loaded 186 RDF triples.


Query all the actors and techniques in the file using SQL.

In [2]:
query_actor_technique = """
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?actorLabel ?techniqueLabel
WHERE {
    ?actor cti:usesTechnique ?technique .
    ?actor rdfs:label ?actorLabel .
    ?technique rdfs:label ?techniqueLabel .
}
"""

for row in g.query(query_actor_technique):
    print(f"{row.actorLabel} uses {row.techniqueLabel}")

APT29 (G0016) uses Spearphishing Attachment (T1566.001)
Elderwood (G0066) uses Spearphishing Attachment (T1566.001)
APT38 (G0082) uses Spearphishing Attachment (T1566.001)
Kimsuky (G0094) uses Spearphishing Attachment (T1566.001)
SideCopy (G1008) uses Spearphishing Attachment (T1566.001)
EXOTIC LILY (G1011) uses Spearphishing Attachment (T1566.001)
APT3 (G0022) uses Scheduled Task (T1053.005)
APT38 (G0082) uses Scheduled Task (T1053.005)
GALLIUM (G0093) uses Scheduled Task (T1053.005)
Kimsuky (G0094) uses Scheduled Task (T1053.005)
BlackByte (G1043) uses Scheduled Task (T1053.005)
APT3 (G0022) uses PowerShell (T1059.001)
APT38 (G0082) uses PowerShell (T1059.001)
GALLIUM (G0093) uses PowerShell (T1059.001)
Kimsuky (G0094) uses PowerShell (T1059.001)
Indrik Spider (G0119) uses PowerShell (T1059.001)
BlackByte (G1043) uses PowerShell (T1059.001)
APT3 (G0022) uses Windows Command Shell (T1059.003)
APT38 (G0082) uses Windows Command Shell (T1059.003)
GALLIUM (G0093) uses Windows Command She

Check all vulnerabilities as well as their corresponding versions and the severity of the risks.

In [4]:
query_vulnerability = """
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?cve ?productLabel ?weaknessLabel
WHERE {
    ?cve rdf:type cti:Vulnerability ;
         cti:affectsProduct ?product ;
         cti:hasWeakness ?weakness .

    ?product rdfs:label ?productLabel .
    ?weakness rdfs:label ?weaknessLabel .
}
"""

for row in g.query(query_vulnerability):
    cve_id = str(row.cve).split("#")[-1]
    print(f"{cve_id} affects {row.productLabel}; weakness: {row.weaknessLabel}")

CVE_2024_30051 affects Windows 11 Version 23H2; weakness: CWE-122: Heap-based Buffer Overflow


Query the available attack techniques and tactics available on Windows.

In [5]:
query_technique_context = """
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?techniqueLabel ?tacticLabel ?platformLabel
WHERE {
    ?technique cti:belongsToTactic ?tactic ;
               cti:applicableToPlatform ?platform .

    ?technique rdfs:label ?techniqueLabel .
    ?tactic rdfs:label ?tacticLabel .
    ?platform rdfs:label ?platformLabel .
}
"""

for row in g.query(query_technique_context):
    print(
        f"Technique: {row.techniqueLabel} | "
        f"Tactic: {row.tacticLabel} | "
        f"Platform: {row.platformLabel}"
    )

Technique: Spearphishing Attachment (T1566.001) | Tactic: Initial Access | Platform: Windows


Draw graph

In [9]:
%pip install networkx matplotlib

  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.


Generate knowledge graph image from the info in the file

In [10]:
tech_query="""
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?technique_name (COUNT(DISTINCT ?actor) AS ?actor_count)
WHERE {
    ?actor a cti:ThreatActor;
        cti:usesTechnique ?technique.

    ?technique rdfs:label ?technique_name.
}
GROUP BY ?technique_name
ORDER BY DESC(?actor_count)
"""
tech_rows = []
for row in g.query(tech_query):
    tech_rows.append({
        "Technique": row.technique_name,
        "Actor Count": row.actor_count
    })
tech_df = pd.DataFrame(tech_rows)
tech_df
tech_df["Actor Count"] = tech_df["Actor Count"].astype(int)
plt.figure(figsize=(9, 5))
plt.bar(range(len(tech_df)), tech_df["Actor Count"], color="steelblue")
plt.xticks(range(len(tech_df)), tech_df["Technique"], rotation=20)
plt.title("Number of Threat Actors Using Each Technique")
plt.xlabel("Technique")
plt.ylabel("Actor Count")
plt.show()

In [ ]:
inte_query="""
    PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?actor_name ?technique_name ?cve_name ?severity_name
    WHERE{
        ?actor a cti:ThreatActor;
            rdfs:label ?actor_name;
            cti:usesTechnique ?technique.
        ?technique rdfs:label ?technique_name;
            cti:applicableToPlatform cti:Windows.
        ?cve a cti:Vulnerability;
            rdfs:label ?cve_name;
            cti:hasSeverity ?severity;
            cti:affectsProduct ?product.
        ?severity rdfs:label ?severity_name.
        ?product cti:belongsToPlatform cti:Windows.
    
    }
    """
inte_row=[]
for row in g.query(inte_query):
    inte_row.append({
        "threat actor": row.actor_name,
        "technique":row.technique_name,
        "CVE":row.cve_name,
        "severity": row.severity_name
    })
integration_df = pd.DataFrame(inte_row)



,threat actor,technique,CVE,severity
0,APT29 (G0016),Spearphishing Attachment (T1566.001),CVE-2024-30051,HIGH severity
1,APT29 (G0016),Spearphishing Attachment (T1566.001),CVE-2024-30051,HIGH severity
2,APT29 (G0016),Spearphishing Attachment (T1566.001),CVE-2024-30051,HIGH severity
3,APT29 (G0016),Spearphishing Attachment (T1566.001),CVE-2024-30051,HIGH severity
4,APT29 (G0016),Spearphishing Attachment (T1566.001),CVE-2024-30051,HIGH severity
...,...,...,...,...
1426,BlackByte (G1043),Windows Command Shell (T1059.003),CVE-2024-38080,HIGH severity
1427,BlackByte (G1043),Windows Command Shell (T1059.003),CVE-2024-38080,HIGH severity
1428,BlackByte (G1043),Windows Command Shell (T1059.003),CVE-2024-38080,HIGH severity
1429,BlackByte (G1043),Windows Command Shell (T1059.003),CVE-2024-38080,HIGH severity
